In [0]:
%sql
USE CATALOG crp_arb_dev;

In [0]:
from pyspark.sql import functions as F

silver_df = spark.readStream.table("crp_arb_dev.silver.crypto_data_24h")

gold_24h_signal = silver_df\
    .withColumn("daily_performance_pct", 
        F.round(((F.col("close") - F.col("open")) / F.col("open")) * 100, 2)) \
    .select(
        "ticker",
        "open",
        "close",
        "daily_performance_pct",
        "trading_volume",
        F.col("high_price").alias("day_high"),
        F.col("low_price").alias("day_low")
    )

container_path = "abfss://crp-arb-cnt@sadlsdev.dfs.core.windows.net"

query_24h = gold_24h_signal.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"{container_path}/_checkpoints/gold_1d") \
    .option("path", f"{container_path}/gold/crypto_data_1d") \
    .trigger(availableNow=True) \
    .toTable("gold.crypto_data_24h")
    
query_24h.awaitTermination()